# 📊 Aggregation & Reporting Tables (SQL → BI Ready)

**Author:** Gabriella Marín  
**Project:** Multi-Layer Water Quality Risk & Regulatory Analytics System     
**Phase:** Phase 5 – Aggregation & Reporting

**Objective:**  
Aggregate sample-level risk indicators into decision-ready reporting tables at multiple
levels (monitoring points, municipalities, and departments), and generate BI-ready
datasets to support visualization, prioritization, and stakeholder communication.

## 1. Paths, DB and Load risk layer

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
PROJECT_DIR = Path(r"D:\Documents\Portfolio\01-water-quality-normative")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

DB_PATH = DATA_DIR / "water_quality_analysis.db"
conn = sqlite3.connect(DB_PATH)

# Helpers
def table_exists(conn, name: str) -> bool:
    q = "SELECT name FROM sqlite_master WHERE type='table' AND name=?"
    return pd.read_sql_query(q, conn, params=[name]).shape[0] > 0

# Load layer risk scores (Phase 4)
risk = pd.read_sql_query("""
SELECT
    r.standard_id,
    r.layer_id,
    r.CODIGO__MUESTRA,
    r.n_params,
    r.violation_score_0_100,
    r.severity_score_0_100,
    r.risk_score_0_100,
    r.critical_violation_flag,
    s.standard_code,
    l.layer_code
FROM fact_layer_risk r
JOIN dim_standard s ON r.standard_id = s.standard_id
JOIN dim_layer l ON r.layer_id = l.layer_id
""", conn)

print("risk rows:", len(risk))
display(risk.head())

# Load sample metadata (prefer SQL; fallback to CSV)
candidate_tables = ["fact_sample", "sample_wide", "fact_sample_wide", "fact_samples"]
sample_tbl = next((t for t in candidate_tables if table_exists(conn, t)), None)

if sample_tbl:
    print("✅ Using sample metadata table from SQL:", sample_tbl)
    sample_meta = pd.read_sql_query(f"SELECT * FROM {sample_tbl};", conn)
else:
    print("⚠️ No sample table found in SQL. Using Phase 1 wide CSV fallback.")
    sample_csv = OUTPUT_DIR / "phase1_sample_wide.csv"
    if not sample_csv.exists():
        raise FileNotFoundError(f"Missing fallback file: {sample_csv}")
    sample_meta = pd.read_csv(sample_csv, parse_dates=["FECHA"], dayfirst=False)

# Keep only needed columns if they exist
wanted_cols = [
    "CODIGO__MUESTRA", "FECHA",
    "NOMBRE DEL PUNTO DE MONITOREO", "DEPARTAMENTO", "MUNICIPIO",
    "LATITUD", "LONGITUD"
]
available = [c for c in wanted_cols if c in sample_meta.columns]
sample_meta = sample_meta[available].copy()

print("sample_meta cols:", list(sample_meta.columns))
display(sample_meta.head())


risk rows: 20446


,standard_id,layer_id,CODIGO__MUESTRA,n_params,violation_score_0_100,severity_score_0_100,risk_score_0_100,critical_violation_flag,standard_code,layer_code
0,1,1,11284,2,57.14,21.43,46.43,1,CO_2115,SANITARY
1,1,1,11285,2,57.14,46.94,54.08,1,CO_2115,SANITARY
2,1,1,11289,2,100.00,55.54,86.66,1,CO_2115,SANITARY
3,1,1,11291,2,57.14,50.79,55.24,1,CO_2115,SANITARY
4,1,1,11292,2,57.14,49.21,54.76,1,CO_2115,SANITARY


✅ Using sample metadata table from SQL: fact_sample
sample_meta cols: ['CODIGO__MUESTRA', 'FECHA', 'NOMBRE DEL PUNTO DE MONITOREO', 'DEPARTAMENTO', 'MUNICIPIO', 'LATITUD', 'LONGITUD']


,CODIGO__MUESTRA,FECHA,NOMBRE DEL PUNTO DE MONITOREO,DEPARTAMENTO,MUNICIPIO,LATITUD,LONGITUD
0,11284,2005-02-15 00:00:00,RCA_BOGOTA_CUN_VILLAPINZON_PTE.CARRETERA-BOGOTA,CUNDINAMARCA,VILLAPINZÓN,5.218611,-73.595556
1,11285,2005-02-15 00:00:00,RCA_BOGOTA_CUN_TOCANCIPA_PTE.TULIO BOTERO-BOGOTA,CUNDINAMARCA,TOCANCIPÁ,4.971917,-73.916139
2,11289,2005-02-15 00:00:00,RCA_BOGOTA_CUN_VILLAPINZON_SANPEDRO-BOGOTA,CUNDINAMARCA,VILLAPINZÓN,5.194722,-73.613889
3,11291,2005-02-16 00:00:00,RCA_BOGOTA_CUN_SOACHA_ALICACHIN - EL SALTO [21...,CUNDINAMARCA,SOACHA,4.544847,-74.258269
4,11292,2005-02-16 00:00:00,RCA_BOGOTA_CUN_SOACHA_ALICACHIN - EL SALTO [21...,CUNDINAMARCA,SOACHA,4.544847,-74.258269


## 2. Build Sample-Level Reporting Table

This section produces a single sample-level table combining:
- metadata (location and site info)
- risk scores per layer and standard

This table is the canonical dataset for BI and reporting.

In [2]:
# Build a canonical sample-level reporting table
# Pivot risk to wide columns by (layer_code, standard_code)
risk_wide = (
    risk.pivot_table(
        index="CODIGO__MUESTRA",
        columns=["layer_code", "standard_code"],
        values="risk_score_0_100",
        aggfunc="mean"
    )
)

# Flatten multiindex columns
risk_wide.columns = [f"risk_{layer}_{std}" for layer, std in risk_wide.columns]
risk_wide = risk_wide.reset_index()

# Critical flags (max over sanitary + discharge)
crit = (risk.groupby("CODIGO__MUESTRA")["critical_violation_flag"].max().reset_index()
        .rename(columns={"critical_violation_flag":"critical_violation_any"}))

# Join metadata + risk
sample_reporting = (
    sample_meta.merge(risk_wide, on="CODIGO__MUESTRA", how="left")
              .merge(crit, on="CODIGO__MUESTRA", how="left")
)

# Fill NA flags
if "critical_violation_any" in sample_reporting.columns:
    sample_reporting["critical_violation_any"] = sample_reporting["critical_violation_any"].fillna(0).astype(int)

display(sample_reporting.head(10))
print("sample_reporting rows:", len(sample_reporting))

# Useful derived fields
# Sanitary "baseline" risk (CO_2115) and EPA benchmark if available
if "risk_SANITARY_CO_2115" in sample_reporting.columns and "risk_SANITARY_EPA" in sample_reporting.columns:
    sample_reporting["sanitary_epa_minus_co"] = (
        sample_reporting["risk_SANITARY_EPA"] - sample_reporting["risk_SANITARY_CO_2115"]
    )

,CODIGO__MUESTRA,FECHA,NOMBRE DEL PUNTO DE MONITOREO,DEPARTAMENTO,MUNICIPIO,LATITUD,LONGITUD,risk_DISCHARGE_CO_0631,risk_SANITARY_CO_2115,risk_SANITARY_EPA,critical_violation_any
0,11284,2005-02-15 00:00:00,RCA_BOGOTA_CUN_VILLAPINZON_PTE.CARRETERA-BOGOTA,CUNDINAMARCA,VILLAPINZÓN,5.218611,-73.595556,0.00,46.43,0.00,1
1,11285,2005-02-15 00:00:00,RCA_BOGOTA_CUN_TOCANCIPA_PTE.TULIO BOTERO-BOGOTA,CUNDINAMARCA,TOCANCIPÁ,4.971917,-73.916139,0.00,54.08,53.71,1
2,11289,2005-02-15 00:00:00,RCA_BOGOTA_CUN_VILLAPINZON_SANPEDRO-BOGOTA,CUNDINAMARCA,VILLAPINZÓN,5.194722,-73.613889,55.28,86.66,87.17,1
3,11291,2005-02-16 00:00:00,RCA_BOGOTA_CUN_SOACHA_ALICACHIN - EL SALTO [21...,CUNDINAMARCA,SOACHA,4.544847,-74.258269,59.06,55.24,55.10,1
4,11292,2005-02-16 00:00:00,RCA_BOGOTA_CUN_SOACHA_ALICACHIN - EL SALTO [21...,CUNDINAMARCA,SOACHA,4.544847,-74.258269,41.78,54.76,54.54,1
5,11294,2005-02-16 00:00:00,RCA_BOGOTA_CUN_EL COLEGIO_PTE.CARRETERA LA MES...,CUNDINAMARCA,EL COLEGIO,4.598969,-74.436010,32.66,56.64,56.63,1
6,11297,2005-02-16 00:00:00,RCA_BOGOTA_CUN_TOCAIMA_PTE.PORTILLO [21207960],CUNDINAMARCA,TOCAIMA,4.454861,-74.608528,53.21,57.00,57.00,1
7,11298,2005-02-17 00:00:00,RCA_MAGDALENA_CUN_GIRARDOT_GIRARDOT [21237030],CUNDINAMARCA,GIRARDOT,4.288073,-74.808592,22.99,56.87,56.87,1
8,11299,2005-02-17 00:00:00,RCA_BOGOTA_CUN_GIRARDOT_CAMPINA LA [21209200],CUNDINAMARCA,GIRARDOT,4.304889,-74.793778,0.00,55.24,55.10,1
9,11300,2005-02-17 00:00:00,RCA_MAGDALENA_TOL_FLANDES_ISLA DEL AMOR,TOLIMA,FLANDES,4.281944,-74.776944,22.98,56.84,56.84,1


sample_reporting rows: 6854


## 3. Aggregations: Points, Municipalities, Departments

This section produces ranked summaries for:
- Monitoring points (sites)
- Municipalities
- Departments

Each summary includes sample counts, mean risk, and critical-flag rates.

In [3]:
# Helper aggregation function
def agg_table(df: pd.DataFrame, group_col: str, sanitary_col: str, discharge_col: str):
    out = df.groupby(group_col).apply(lambda g: pd.Series({
        "n_samples": int(g["CODIGO__MUESTRA"].nunique()),
        "sanitary_mean": float(g[sanitary_col].mean()) if sanitary_col in g else np.nan,
        "discharge_mean": float(g[discharge_col].mean()) if discharge_col in g else np.nan,
        "critical_rate_pct": float(g["critical_violation_any"].mean()*100) if "critical_violation_any" in g else np.nan,
    })).reset_index()
    return out

# Choose columns (they may be missing if norms not included yet)
SAN_BASE = "risk_SANITARY_CO_2115"
DIS_BASE = "risk_DISCHARGE_CO_0631"

# If discharge column name differs, try to find it
if DIS_BASE not in sample_reporting.columns:
    # fallback: any discharge col
    dis_candidates = [c for c in sample_reporting.columns if c.startswith("risk_DISCHARGE_")]
    if dis_candidates:
        DIS_BASE = dis_candidates[0]

# Point-level summary
if "NOMBRE DEL PUNTO DE MONITOREO" in sample_reporting.columns:
    point_summary = agg_table(sample_reporting, "NOMBRE DEL PUNTO DE MONITOREO", SAN_BASE, DIS_BASE)

    # Add centroid lat/lon if available
    for col in ["LATITUD", "LONGITUD", "DEPARTAMENTO", "MUNICIPIO"]:
        if col in sample_reporting.columns:
            add = (sample_reporting.groupby("NOMBRE DEL PUNTO DE MONITOREO")[col]
                   .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
                   .reset_index())
            point_summary = point_summary.merge(add, on="NOMBRE DEL PUNTO DE MONITOREO", how="left")

    # Rank
    point_summary["sanitary_rank"] = point_summary["sanitary_mean"].rank(method="dense", ascending=False)
    point_summary["discharge_rank"] = point_summary["discharge_mean"].rank(method="dense", ascending=False)

    display(point_summary.sort_values("sanitary_mean", ascending=False).head(20))
else:
    point_summary = pd.DataFrame()
    print("⚠️ No point name column found. Skipping point aggregation.")

# Municipality summary
if "MUNICIPIO" in sample_reporting.columns:
    muni_summary = agg_table(sample_reporting, "MUNICIPIO", SAN_BASE, DIS_BASE)
    if "DEPARTAMENTO" in sample_reporting.columns:
        dep_map = (sample_reporting.groupby("MUNICIPIO")["DEPARTAMENTO"]
                   .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) else np.nan)
                   .reset_index())
        muni_summary = muni_summary.merge(dep_map, on="MUNICIPIO", how="left")

    muni_summary["sanitary_rank"] = muni_summary["sanitary_mean"].rank(method="dense", ascending=False)
    display(muni_summary.sort_values("sanitary_mean", ascending=False).head(20))
else:
    muni_summary = pd.DataFrame()
    print("⚠️ No municipality column found. Skipping municipality aggregation.")

# Department summary
if "DEPARTAMENTO" in sample_reporting.columns:
    dept_summary = agg_table(sample_reporting, "DEPARTAMENTO", SAN_BASE, DIS_BASE)
    dept_summary["sanitary_rank"] = dept_summary["sanitary_mean"].rank(method="dense", ascending=False)
    display(dept_summary.sort_values("sanitary_mean", ascending=False).head(20))
else:
    dept_summary = pd.DataFrame()
    print("⚠️ No department column found. Skipping department aggregation.")

# Sanitary vs Discharge risk matrix (sample-level)
# Define risk bands (tunable)
def band(x):
    if pd.isna(x): return "NA"
    if x < 20: return "Low"
    if x < 50: return "Medium"
    return "High"

matrix = sample_reporting.copy()
if SAN_BASE in matrix.columns:
    matrix["sanitary_band"] = matrix[SAN_BASE].apply(band)
else:
    matrix["sanitary_band"] = "NA"

if DIS_BASE in matrix.columns:
    matrix["discharge_band"] = matrix[DIS_BASE].apply(band)
else:
    matrix["discharge_band"] = "NA"

risk_matrix = (
    matrix.groupby(["sanitary_band","discharge_band"])["CODIGO__MUESTRA"]
          .nunique()
          .reset_index(name="n_samples")
)

display(risk_matrix.sort_values("n_samples", ascending=False).head(20))

,NOMBRE DEL PUNTO DE MONITOREO,n_samples,sanitary_mean,discharge_mean,critical_rate_pct,LATITUD,LONGITUD,DEPARTAMENTO,MUNICIPIO,sanitary_rank,discharge_rank
102,RCA_CAUCA_CAU_POPAYAN_PUENTE LA VARIANTE,1.0,87.480000,0.000000,100.000000,2.463006,-76.634722,CAUCA,POPAYÁN,1.0,214.0
101,RCA_CAUCA_CAU_POPAYAN_JULUMITO [26017020],32.0,84.887500,8.173438,100.000000,2.467583,-76.644306,CAUCA,POPAYÁN,2.0,150.0
149,RCA_INIRIDA_GUAI_INIRIDA_PTO.INIRIDA [31097020],34.0,79.797059,0.000000,100.000000,3.867667,-67.931694,GUAINÍA,INÍRIDA,3.0,214.0
16,CÑO.SAN SILVESTRE_SAN_BARRANCABERMEJA_PTE.FERR...,8.0,77.501250,0.000000,100.000000,7.109444,-73.854722,SANTANDER,BARRANCABERMEJA,4.0,214.0
188,RCA_OCOA_MET_VILLAVICENCIO_PTE.DELAMOR [35037130],37.0,75.851081,0.892162,100.000000,4.086083,-73.668861,META,VILLAVICENCIO,5.0,212.0
228,RCA_VAUPES_VAU_MITU_MITU [42077020],33.0,75.158788,0.000000,96.969697,1.259806,-70.239167,VAUPES,MITÚ,6.0,214.0
191,RCA_ORINOCO_VIC_PUERTO CARRENO_PTO.CARRENO [38...,38.0,73.388205,0.000000,100.000000,6.176528,-67.479583,VICHADA,PUERTO CARREÑO,7.0,214.0
93,RCA_CASANARE_ARA_CRAVO NORTE_CRAVONORTE [36027...,32.0,70.896667,21.891212,100.000000,6.295639,-70.194333,ARAUCA,CRAVO NORTE,8.0,63.0
230,RCA_VICHADA_VIC_CUMARIBO_STA.RITA [33077010],30.0,68.651000,0.000000,100.000000,4.866127,-68.362697,VICHADA,CUMARIBO,9.0,214.0
112,RCA_CGA.ZAPATOZA_CES_CHIMICHAGUA_SALOA [25027140],5.0,68.568000,6.646000,100.000000,9.249722,-73.806667,CESAR,CHIMICHAGUA,10.0,159.0


,MUNICIPIO,n_samples,sanitary_mean,discharge_mean,critical_rate_pct,DEPARTAMENTO,sanitary_rank
102,POPAYÁN,33.0,84.966061,7.925758,100.000000,CAUCA,1.0
60,INÍRIDA,34.0,79.797059,0.000000,100.000000,GUAINÍA,2.0
80,MITÚ,33.0,75.158788,0.000000,96.969697,VAUPES,3.0
31,CRAVO NORTE,32.0,70.896667,21.891212,100.000000,ARAUCA,4.0
105,PUERTO CARREÑO,76.0,69.583077,14.350513,100.000000,VICHADA,5.0
22,CHIMICHAGUA,5.0,68.568000,6.646000,100.000000,CESAR,6.0
66,LENGUAZAQUE,52.0,67.753019,1.781509,100.000000,CUNDINAMARCA,7.0
117,RIOHACHA,3.0,66.286667,11.043333,100.000000,LA GUAJIRA,8.0
77,MAPIRIPÁN,28.0,66.185357,25.807143,100.000000,META,9.0
24,CHIRIGUANÁ,7.0,65.911429,9.454286,100.000000,CESAR,10.0


,DEPARTAMENTO,n_samples,sanitary_mean,discharge_mean,critical_rate_pct,sanitary_rank
14,GUAINÍA,34.0,79.797059,0.000000,100.000000,1.0
26,VAUPES,33.0,75.158788,0.000000,96.969697,2.0
2,ARAUCA,32.0,70.896667,21.891212,100.000000,3.0
9,CAUCA,77.0,70.481169,10.623377,100.000000,4.0
27,VICHADA,162.0,67.306121,10.077697,98.787879,5.0
21,PUTUMAYO,40.0,65.203250,5.366000,100.000000,6.0
11,CHOCÓ,40.0,63.936341,17.267317,95.121951,7.0
25,VALLE DEL CAUCA,92.0,63.236774,16.506989,100.000000,8.0
0,AMAZONAS,67.0,61.864179,24.919104,100.000000,9.0
18,META,248.0,61.271084,20.831406,98.795181,10.0


,sanitary_band,discharge_band,n_samples
1,High,Low,2715
2,High,Medium,2666
7,Medium,Low,790
4,Low,Low,326
0,High,High,247
5,Low,Medium,53
3,Low,High,13
8,Medium,Medium,3
9,NA,Low,3
6,Low,NA,2


## 4. Export Deliverables

This section exports BI-ready CSV tables:
- sample-level reporting dataset
- point, municipality, and department summaries
- sanitary vs discharge risk matrix

In [4]:
# Export CSV deliverables
sample_out = OUTPUT_DIR / "phase5_sample_reporting.csv"
sample_reporting.to_csv(sample_out, index=False)

if not point_summary.empty:
    point_out = OUTPUT_DIR / "phase5_point_risk_summary.csv"
    point_summary.to_csv(point_out, index=False)

if not muni_summary.empty:
    muni_out = OUTPUT_DIR / "phase5_municipality_risk_summary.csv"
    muni_summary.to_csv(muni_out, index=False)

if not dept_summary.empty:
    dept_out = OUTPUT_DIR / "phase5_department_risk_summary.csv"
    dept_summary.to_csv(dept_out, index=False)

matrix_out = OUTPUT_DIR / "phase5_sanitary_vs_discharge_matrix.csv"
risk_matrix.to_csv(matrix_out, index=False)

print("✅ Exported:")
print("-", sample_out)
if not point_summary.empty: print("-", point_out)
if not muni_summary.empty: print("-", muni_out)
if not dept_summary.empty: print("-", dept_out)
print("-", matrix_out)

# Optional: store as SQL tables for Power BI direct connection
# (Safe replacements; comment out if you prefer only CSV)
sample_reporting.to_sql("report_sample", conn, if_exists="replace", index=False)
if not point_summary.empty:
    point_summary.to_sql("report_point", conn, if_exists="replace", index=False)
if not muni_summary.empty:
    muni_summary.to_sql("report_municipality", conn, if_exists="replace", index=False)
if not dept_summary.empty:
    dept_summary.to_sql("report_department", conn, if_exists="replace", index=False)
risk_matrix.to_sql("report_risk_matrix", conn, if_exists="replace", index=False)

print("✅ Reporting tables stored in SQL: report_*")


✅ Exported:
- D:\Documents\Portfolio\01-water-quality-normative\outputs\phase5_sample_reporting.csv
- D:\Documents\Portfolio\01-water-quality-normative\outputs\phase5_point_risk_summary.csv
- D:\Documents\Portfolio\01-water-quality-normative\outputs\phase5_municipality_risk_summary.csv
- D:\Documents\Portfolio\01-water-quality-normative\outputs\phase5_department_risk_summary.csv
- D:\Documents\Portfolio\01-water-quality-normative\outputs\phase5_sanitary_vs_discharge_matrix.csv
✅ Reporting tables stored in SQL: report_*


## Phase 5 Summary

This phase transformed sample-level risk outputs into BI-ready reporting tables.

Risk scores were aggregated across multiple decision levels (monitoring points, municipalities, and departments),
including sample counts and critical-flag rates to support prioritization.

A sanitary vs discharge risk matrix was also produced to distinguish sites with public health risk from those
primarily driven by environmental/operational pressure.

All deliverables were exported as clean CSV tables (and optionally stored as SQL `report_*` tables) to enable
dashboarding and stakeholder-friendly communication.